In [ ]:
# prompt: mount drive

from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install wandb -qU
import wandb
wandb.login()


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sblas (sblas-universidad-nacional-del-litoral) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
sweep_config = {
    'method': 'bayes',  # Puede ser random, grid o bayes
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'latent_dim': {
            'values': [128, 250, 512]
        },
        'hidden_dim': {
            'values': [512, 1024, 2048, 3072, 4096, 5120]
        },
        'beta': {
            'distribution': 'uniform',
            'min': 0.5,
            'max': 5.0
        },
        'dropout': {
            'values': [0.0, 0.01, 0.1]
        },
        'lr': {
            'distribution': 'log_uniform_values',
            'min': 1e-5,
            'max': 1e-2
        },
        'batch_size': {
            'values': [16, 32, 64]
        },
        'weight_decay': {
            'distribution': 'log_uniform_values',
            'min': 1e-8,
            'max': 1e-4
        }
    }
}

import pprint
pprint.pprint(sweep_config)


{'method': 'bayes',
 'metric': {'goal': 'minimize', 'name': 'val_loss'},
 'parameters': {'batch_size': {'values': [16, 32, 64]},
                'beta': {'distribution': 'uniform', 'max': 5.0, 'min': 0.5},
                'dropout': {'values': [0.0, 0.01, 0.1]},
                'hidden_dim': {'values': [512, 1024, 2048, 3072, 4096, 5120]},
                'latent_dim': {'values': [128, 250, 512]},
                'lr': {'distribution': 'log_uniform_values',
                       'max': 0.01,
                       'min': 1e-05},
                'weight_decay': {'distribution': 'log_uniform_values',
                                 'max': 0.0001,
                                 'min': 1e-08}}}


In [ ]:
sweep_id = wandb.sweep(sweep_config, project="vae-hyperparam-search")


Create sweep with ID: m6el4he6
Sweep URL: https://wandb.ai/sblas-universidad-nacional-del-litoral/vae-hyperparam-search/sweeps/m6el4he6


In [ ]:
# Función de pérdida
def loss_function(recon_x, x, mu, logvar, beta):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    total_loss = recon_loss + beta * kld
    return total_loss, recon_loss.item(), kld.item()

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import os
import sys
# Get the absolute path to the directory containing 'models'
# Assuming your 'models' directory is in '/content/drive/MyDrive/1st_paper/'
models_dir = os.path.join('/content/drive/MyDrive/1st_paper/')

# Append it to sys.path
sys.path.append(models_dir)

# The rest of your code from ipython-input-6-b9d3ab608c4b
from models.vae import BetaVAE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def train():
    # Inicializa W&B run
    wandb.init()

    config = wandb.config

    # Carga tus datos (modifica con tu path correcto)
    train_data = torch.load('/content/drive/MyDrive/1st_paper/fold_1/train_data_normed.pt')
    val_data = torch.load('/content/drive/MyDrive/1st_paper/fold_1/val_data_normed.pt')

    train_loader = DataLoader(TensorDataset(train_data), batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(val_data), batch_size=config.batch_size)

    # Inicializa modelo con hiperparámetros del Sweep
    model = BetaVAE(config.latent_dim, config.hidden_dim, config.beta, config.dropout, input_channels=4).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5)

    best_val_loss = float('inf')

    epochs = 200  # Puedes ajustar esto o añadirlo al sweep_config

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for x, in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            recon, mu, logvar, _ = model(x)
            loss, _, _ = loss_function(recon, x, mu, logvar, config.beta)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)

        train_loss /= len(train_loader.dataset)

        # Evalúa
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, in val_loader:
                x = x.to(device)
                recon, mu, logvar, _ = model(x)
                loss, _, _ = loss_function(recon, x, mu, logvar, config.beta)
                val_loss += loss.item() * x.size(0)

        val_loss /= len(val_loader.dataset)

        # Scheduler
        scheduler.step(val_loss)

        # Guarda modelo si es mejor
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f'/content/drive/MyDrive/1st_paper/wandb_best_model.pth')

        # Loguear métricas en W&B
        wandb.log({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss
        })


Using device: cuda


In [ ]:
wandb.agent(sweep_id, train, count=50)  # Realiza 20 experimentos diferentes


wandb: Agent Starting Run: sv3g6n6a with config:
wandb: 	batch_size: 64
wandb: 	beta: 4.437270029857666
wandb: 	dropout: 0
wandb: 	hidden_dim: 2048
wandb: 	latent_dim: 128
wandb: 	lr: 3.2113963120012165e-05
wandb: 	weight_decay: 1.9862461800801572e-08


epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█████
train_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▇▇█████████████████████████████████████
epoch,199
train_loss,115.76208
val_loss,1463.38611


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: l2ttjtfh with config:
wandb: 	batch_size: 32
wandb: 	beta: 2.3547508300520814
wandb: 	dropout: 0
wandb: 	hidden_dim: 2048
wandb: 	latent_dim: 512
wandb: 	lr: 0.002330676388493474
wandb: 	weight_decay: 4.14884662051181e-07


epoch,▁▁▁▁▂▂▃▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇█████
train_loss,█▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,199
train_loss,1.22006
val_loss,1.22505


wandb: Agent Starting Run: 3hp4u4fl with config:
wandb: 	batch_size: 32
wandb: 	beta: 0.5808120877155127
wandb: 	dropout: 0.1
wandb: 	hidden_dim: 5120
wandb: 	latent_dim: 512
wandb: 	lr: 0.0067944223936383455
wandb: 	weight_decay: 1.8206940100160403e-06


...

epoch,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▁▂▃▅▆███████████████████████████████████
epoch,199
train_loss,253.51988
val_loss,680.47174


wandb: Agent Starting Run: 5fyftnzn with config:
wandb: 	batch_size: 16
wandb: 	beta: 1.3865344212438755
wandb: 	dropout: 0
wandb: 	hidden_dim: 512
wandb: 	latent_dim: 512
wandb: 	lr: 1.8254246533831017e-05
wandb: 	weight_decay: 1.846316007562404e-08


epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇████
train_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▇▄▃▄▃▃▃▂▂▄▃▁▂▃▃▃▁▃▁▃▃▃▂▃▄▃▃▂▁▄▁▁▅▄▃▂▂▄▃
epoch,199
train_loss,343.58907
val_loss,446.5123


wandb: Agent Starting Run: 225kgbh7 with config:
wandb: 	batch_size: 16
wandb: 	beta: 2.2377425831445255
wandb: 	dropout: 0
wandb: 	hidden_dim: 512
wandb: 	latent_dim: 250
wandb: 	lr: 3.518947215146432e-05
wandb: 	weight_decay: 5.119038164532969e-06


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█
train_loss,█▃▆▅▅▃▄▂▆▄▄▇▆▃▂▆▃▄█▇▄▅▃▃▄▂▄▇▄▃▂▃▆▆▃▅▃▆▂▁
val_loss,█▂▂▂▁▁▁▁▁▂▁▂▂▂▂▁▁▁▂▁▁▁▁▁▁▁▁▂▁▁▂▁▁▂▁▁▁▁▁▁
epoch,199
train_loss,101.79182
val_loss,175.08343
